# Telugu and Nepali GPT: live demo
Two 17M-parameter GPT models trained from scratch on Wikipedia.
Code: https://github.com/Yogesh-Narasimha/telugu-nepali-lm

Run all cells (Runtime > Run all). A GPU is not needed. Text is fluent but often factually wrong.

In [ ]:
!pip -q install sentencepiece huggingface_hub

In [ ]:
import torch, sentencepiece as spm
from huggingface_hub import hf_hub_download

REPOS = {"Telugu": "YogeshNarasimha/telugu-gpt-17m", "Nepali": "YogeshNarasimha/nepali-gpt-17m"}

# model.py ships inside each model repo
import importlib.util, shutil
src = hf_hub_download(REPOS["Telugu"], "model.py")
shutil.copy(src, "model.py")
from model import GPT, GPTConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
loaded = {}
for name, repo in REPOS.items():
    ckpt = torch.load(hf_hub_download(repo, "model.pt"), map_location=device)
    model = GPT(GPTConfig(**ckpt["config"])).to(device).eval()
    model.load_state_dict(ckpt["model"])
    sp = spm.SentencePieceProcessor(model_file=hf_hub_download(repo, "tokenizer.model"))
    loaded[name] = (model, sp)
print("loaded:", list(loaded))

In [ ]:
def generate(language, prompt, tokens=80, temperature=0.8, top_k=40, seed=0):
    model, sp = loaded[language]
    torch.manual_seed(seed)
    ids = torch.tensor([sp.encode(prompt)], device=device)
    out = model.generate(ids, tokens, temperature, top_k)[0].tolist()
    return sp.decode([t for t in out if t != sp.eos_id()])

print(generate("Telugu", "భారతదేశం"))
print()
print(generate("Nepali", "नेपाल एक"))

Try your own prompt (change the values and re-run):

In [ ]:
language = "Nepali"   #@param ["Telugu", "Nepali"]
prompt = "काठमाडौं उपत्यका"   #@param {type:"string"}
tokens = 80   #@param {type:"slider", min:10, max:200, step:10}
temperature = 0.8   #@param {type:"slider", min:0.3, max:1.5, step:0.05}
print(generate(language, prompt, tokens, temperature))